In [1]:
import pandas as pd
import ast
import numpy as np
import difflib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Denise\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### Funzioni generali
def clean_json(json_text):
    list_names = []
    converted_list = ast.literal_eval(json_text)
    for item in converted_list:
        list_names.append(item["name"])
    return list_names

def cleaning_and_preprocessing(df):
    df.dropna(inplace=True)
    df["release_date"] = pd.to_datetime(df["release_date"], errors='coerce')
    df["release_date"] = df["release_date"].dt.strftime('%d/%m/%Y')
    df.rename(columns={"title_x": "title"}, inplace=True)
    df["genres"] = df["genres"].apply(clean_json)
    df["genres"] = df["genres"].apply(lambda x: [genere.lower() for genere in x])
    df["keywords"] = df["keywords"].apply(clean_json)
    df["cast"] = df["cast"].apply(clean_json)
    df["crew"] = df["crew"].apply(clean_json)
    df["production_companies"] = df["production_companies"].apply(clean_json)
    df["production_countries"] = df["production_countries"].apply(clean_json)
    df["spoken_languages"] = df["spoken_languages"].apply(clean_json)
    df.drop(["id", "movie_id", "title_y", "budget", "revenue", "production_countries"], axis=1, inplace=True)
    return df

def join_list_elements(dataframe, columns):
    for col in columns:
        dataframe[col] = dataframe[col].apply(lambda x: " ".join(x))
    return dataframe

def find_genres(user_query, unique_genres):
    query_lowered = user_query.lower()
    genres_found = [gen for gen in unique_genres if gen in query_lowered]
    return genres_found

In [ ]:
### Preprocessing e Pulizia
movies = pd.read_csv("Dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("Dataset/tmdb_5000_credits.csv")
df = pd.merge(movies, credits, left_on="id", right_on="movie_id")

df = cleaning_and_preprocessing(df)
df.to_csv('tmdb_5000_pulito.csv', index=False)
print("Dataset salvato con successo!")

In [ ]:
### Testo unico per la Sentence
df = join_list_elements(df, ["cast", "crew", "keywords", "production_companies"])

### Lista Generi
all_genres = [genres for sublist in df["genres"] for genres in sublist]
unique_genres = sorted(list(set(all_genres)))
#print(f"Generi individuati:{unique_genres}")


### Sentence Embedding e Cosine Similarity
sentence_var = (df["overview"] + " " + df["cast"] + " " + df["crew"] + " " + df["keywords"] + " " + df["tagline"] +" " + df["production_companies"]) 
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_sentence = model.encode(sentence_var.tolist(), convert_to_numpy=True)
#similarity_matrix = cosine_similarity(embeddings_sentence)

In [ ]:
### Sentence
## Calcolo score sentence ed embedding della query
def calc_sentence_score(user_query):
    query_embedding = model.encode([user_query], convert_to_numpy=True)
    sentence_cosine = cosine_similarity(query_embedding, embeddings_sentence)[0]
    # Valore da -1 a 1, embedding forza 0 a 1
    # Normalizzazione
    sentence_score_normalized = (sentence_cosine - sentence_cosine.min()) / (sentence_cosine.max()-sentence_cosine.min())
    return sentence_score_normalized


### Genres
## Findings dei generi presenti nella query
def find_genres(user_query, unique_genres):
    query_lowered = user_query.lower()
    genres_found = [gen for gen in unique_genres if gen in query_lowered]
    return genres_found

## Confronto tra i generi nella query e i film per genere
def calc_genres_score(query_genres):
    genres_score = []
    for genres in df["genres"]:
        if len(query_genres) == 0:
            genres_score.append(0)
        else:
            intersection = set(query_genres) & set(genres)
            num_genres = len(intersection) / len (query_genres)
            genres_score.append(num_genres)
    return genres_score


### Title
## Ricerca tramite titolo corretto
def find_exact_title(user_query, df):
    query_normalized = user_query.lower()
    title_normalized = df["title"].str.lower()
    title_found = [titolo for titolo in title_normalized if query_normalized in titolo]
    return title_found

## Ricerca titolo con difflib in caso di typo o parziali
def find_typos_title(user_query, df):
    query_normalized = user_query.lower()
    title_normalized = df["title"].str.lower().tolist()
    typos_title = difflib.get_close_matches(query_normalized, title_normalized)
    return typos_title

## Ricerca titolo se acronimo
def find_acronym_title(user_query, df):
    query_normalized = user_query.lower()
    title_normalized = df["title"].str.lower().str.split()
    
    acro_title = "".join([acro[0] for acro in title_normalized for acro in query_normalized])
    return acro_title
    
## Calcolo score titolo
def calc_title_score(title_found, typos_title, acro_title):
    title_score = 
    return title_score


### Calcolo score totale con weight
def calc_total_score(sentence_score_normalized, genres_score):
    total_score = (np.array(sentence_score_normalized) * 0.3) + (np.array(genres_score) * 0.7)
    return total_score

### Funzione main
def search_film():
    user_query = input("Inserisci la tua query: ")
    # Encoding della query e calcolo del peso
    sentence_score = calc_sentence_score(user_query)
    # Findings dei generi nella query e calcolo del peso
    query_genres = find_genres(user_query, unique_genres)
    genres_score = calc_genres_score(query_genres)
    total_score = calc_total_score(sentence_score, genres_score)
    results = list(enumerate(total_score))
    top_10 = sorted(results, key=lambda x: x[1], reverse=True)[:10]
    index = [i[0] for i in top_10]
    table = df.iloc[index][["title", "release_date", "genres", "spoken_languages", "homepage"]]
    print("\nFilm Consigliati per te")
    return table


search_film()